# Infrastructure monitoring with OpenTelemetry: 3 collector layouts for a 20-node cluster and what each costs to store

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/infrastructure-monitoring-opentelemetry-collector-cost).
It creates a 20-node [kind](https://kind.sigs.k8s.io/) cluster, runs each collector layout at each
interval, measures the results in Elastic and cleans up at the end.

Requires Elastic Cloud Hosted 9.5 with the managed OTLP endpoint, Docker with 21 GiB and 10 CPUs
(`WORKERS="5"` fits in 8 GiB), kind, `kubectl`, Helm, and a `.env` copied from `.env.example`.
One run (`LAB_RUNS="2:60"`) takes about 25 minutes, all nine (`LAB_RUNS="all"`) about 3.5 hours.

Create the collectors' ingest key in Kibana **Dev Tools** and copy its `encoded` value to
`ELASTIC_INGEST_API_KEY`:

```
POST /_security/api_key
{
  "name": "infrastructure-monitoring-opentelemetry-collector-cost-ingest",
  "expiration": "7d",
  "role_descriptors": {
    "otlp_ingest": {
      "cluster": ["monitor"],
      "indices": [{ "names": ["logs-*", "metrics-*", "traces-*"], "privileges": ["create_doc", "auto_configure"] }],
      "applications": [{ "application": "apm", "privileges": ["event:write"], "resources": ["*"] }]
    }
  }
}
```

As a superuser, create the notebook's key and copy its `encoded` value to `ELASTICSEARCH_API_KEY`:

```
POST /_security/api_key
{ "name": "infrastructure-monitoring-opentelemetry-collector-cost-notebook", "expiration": "7d", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

Loads `.env` and stops if the kind cluster or the run data streams already exist.

In [ ]:
import json
import os
import re
import secrets
import shutil
import statistics
import subprocess
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, ElasticsearchWarning
from elasticsearch.exceptions import GeneralAvailabilityWarning

load_dotenv()
# _disk_usage is a technical preview API; the cluster settings call lists deprecated defaults
warnings.filterwarnings("ignore", category=GeneralAvailabilityWarning)
warnings.filterwarnings("ignore", category=ElasticsearchWarning)

ES_URL = os.environ["ELASTICSEARCH_URL"].rstrip("/")
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
OTLP_ENDPOINT = os.environ["ELASTIC_OTLP_ENDPOINT"]
INGEST_API_KEY = os.environ["ELASTIC_INGEST_API_KEY"]

KIND_CLUSTER = os.getenv("KIND_CLUSTER", "infra-cost")
WORKERS = int(
    os.getenv("WORKERS", "20")
)  # the article used 20; 10 shop pods per worker either way
KCTX = f"kind-{KIND_CLUSTER}"
NODE_IMAGE = "kindest/node:v1.35.0"
NS = "opentelemetry-operator-system"
CHART_VERSION = "0.16.0"
EDOT_VALUES = (
    "https://raw.githubusercontent.com/elastic/elastic-agent/refs/tags/v9.5.5/"
    "deploy/helm/edot-collector/kube-stack/managed_otlp/values.yaml"
)
LAYOUT_FILES = {
    1: ["values/layout1-daemonset-only.yaml"],
    2: [],
    3: ["values/layout3-gateway-filter.yaml"],
}  # layout 2 = EDOT defaults
LAYOUT_NAMES = {1: "DaemonSet only", 2: "DaemonSet + gateway", 3: "Gateway + filter"}
DATASETS = ["hostmetricsreceiver.otel", "kubeletstatsreceiver.otel"]
WARMUP = int(os.getenv("WARMUP_SECONDS", "180"))
MEASURE = int(os.getenv("MEASURE_SECONDS", "720"))
LATENCY_REPS = int(os.getenv("LATENCY_REPS", "15"))
KEEP_KIND_CLUSTER = os.getenv("KEEP_KIND_CLUSTER", "false").lower() == "true"
RUN_SYNTHETIC = os.getenv("RUN_SYNTHETIC", "false").lower() == "true"
SYN_PORT = int(os.getenv("SYNTHETIC_PORT", "9200"))
SYN_CONTAINER = "es-synthetic-collector-cost"

# The article's order. It never upgrades from layout 3 straight to layout 2, which Helm 4
# rejects (spec.config.processors: null).
FULL_SWEEP = "3:60,3:30,3:10,1:60,1:30,1:10,2:10,2:30,2:60"
SUFFIX = os.getenv("RUN_SUFFIX", "")
if not re.fullmatch(r"[a-z0-9]*", SUFFIX):
    raise ValueError("RUN_SUFFIX must be lowercase letters and digits only")
lab_runs = os.getenv("LAB_RUNS", "2:60").strip()
lab_runs = FULL_SWEEP if lab_runs == "all" else lab_runs
RUNS = []  # (run id = data stream namespace, layout, interval in seconds)
for item in lab_runs.split(","):
    layout, interval = (int(x) for x in item.strip().split(":"))
    RUNS.append((f"l{layout}s{interval}{SUFFIX}", layout, interval))
RUN_IDS = [r for r, _, _ in RUNS]
for (_, a, _), (_, b, _) in zip(RUNS, RUNS[1:]):
    if a == 3 and b == 2:
        raise ValueError(
            "LAB_RUNS goes from layout 3 straight to layout 2; put a layout 1 run in between"
        )

HERE = Path.cwd()
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
slow = es.options(request_timeout=1800)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
    "x-elastic-internal-origin": "Kibana",
}


def sh(cmd, check=True, quiet=False, timeout=None, input=None):
    """Run a command (list, or string for a shell pipeline), print its output, return stdout."""
    try:
        p = subprocess.run(
            cmd,
            shell=isinstance(cmd, str),
            text=True,
            capture_output=True,
            timeout=timeout,
            input=input,
        )
    except subprocess.TimeoutExpired:
        if check:
            raise RuntimeError(
                f"command timed out after {timeout}s: {cmd if isinstance(cmd, str) else ' '.join(cmd)}"
            )
        print(f"timed out after {timeout}s, continuing")
        return ""
    out = (p.stdout + p.stderr).strip()
    if out and not quiet:
        print(out[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(
            f"command failed ({p.returncode}): {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n{out[-2000:]}"
        )
    return p.stdout


def kubectl(*args, retries=6, **kw):
    """kubectl on the lab cluster. Retries when the API server is briefly unreachable, which
    happens on a busy laptop (TLS handshake timeout); other errors are raised at once.
    """
    for attempt in range(retries):
        try:
            return sh(["kubectl", "--context", KCTX, *args], **kw)
        except RuntimeError as e:
            if attempt == retries - 1 or not re.search(
                r"TLS handshake timeout|Unable to connect to the server|connection refused",
                str(e),
            ):
                raise
            time.sleep(15)


def helm(*args, **kw):
    return sh(["helm", "--kube-context", KCTX, *args], **kw)


def esql(query, title=None):
    """Run an ES|QL query and print the result as a text table."""
    response = es.esql.query(query=query, format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)
    return response.body


def esql_rows(query):
    """Run an ES|QL query and return a list of dicts."""
    res = es.esql.query(query=query)
    cols = [c["name"] for c in res["columns"]]
    return [dict(zip(cols, row)) for row in res["values"]]


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_iso(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


def window(run):
    d = RESULTS / run
    return (d / "t0").read_text().strip(), (d / "t1").read_text().strip()


def pct(a, b):
    """Relative change from b to a, in percent."""
    return round((a / b - 1) * 100, 1)


# Tools and Docker resources
missing = [t for t in ("kind", "kubectl", "helm", "docker") if not shutil.which(t)]
if missing:
    raise RuntimeError(f"missing on PATH: {', '.join(missing)}")
mem, cpus = sh(
    ["docker", "info", "--format", "{{.MemTotal}} {{.NCPU}}"], quiet=True
).split()
print(f"docker: {int(mem) / 2**30:.1f} GiB, {cpus} CPUs | kind workers: {WORKERS}")
if int(mem) < WORKERS * 2**30 or int(cpus) < min(10, WORKERS // 2):
    print(
        f"WARNING: {WORKERS} kind workers need about {WORKERS + 1} GiB and {min(10, WORKERS // 2)} CPUs; collectors may fail their health checks"
    )

# Stack
info = es.info()["version"]["number"]
license_type = es.license.get()["license"]["type"]
print(f"elasticsearch {info}, license {license_type} | space={SPACE}")
print(f"runs: {', '.join(RUN_IDS)} | warm-up {WARMUP}s, measure {MEASURE}s per run")

# Nothing this notebook creates may already exist, unless an earlier run of this notebook
# created it (results/ keeps a marker for the kind cluster and a folder per started run).
KIND_MARKER = RESULTS / f"kind-{KIND_CLUSTER}.created"
if (
    KIND_CLUSTER in sh(["kind", "get", "clusters"], quiet=True).split()
    and not KIND_MARKER.exists()
):
    raise RuntimeError(
        f"kind cluster {KIND_CLUSTER} already exists: delete it (kind delete cluster --name {KIND_CLUSTER}) or set KIND_CLUSTER"
    )
taken = {}
for run in RUN_IDS:
    if (RESULTS / run / "run-values.yaml").exists():
        continue  # started by this notebook
    streams = (
        es.options(ignore_status=404)
        .indices.get_data_stream(name=f"*-{run}")
        .body.get("data_streams", [])
    )
    if streams:
        taken[run] = [s["name"] for s in streams]
if taken:
    raise RuntimeError(
        f"run namespaces already have data, set RUN_SUFFIX or delete them first: {json.dumps(taken, indent=1)}"
    )

# Shard headroom: five data streams per run, one primary and up to one replica each
health = es.cluster.health()
cs = es.cluster.get_settings(include_defaults=True, flat_settings=True)
limit = int(
    {**cs["defaults"], **cs["persistent"], **cs["transient"]}.get(
        "cluster.max_shards_per_node", 1000
    )
)
capacity = limit * health["number_of_data_nodes"]
used = health["active_shards"] + health["unassigned_shards"]
needed = 10 * len(RUNS)
print(
    f"shards: {used} used + {needed} for this lab, limit {capacity} ({limit} per node x {health['number_of_data_nodes']} data nodes)"
)
if used + needed > capacity:
    raise RuntimeError("not enough shard headroom: delete old data streams first")

## 2. Create the 20-node kind cluster and the workload

1 control plane, `WORKERS` workers and 10 `pause` pods per worker.

In [ ]:
if KIND_CLUSTER in sh(["kind", "get", "clusters"], quiet=True).split():
    print(
        f"reusing kind cluster {KIND_CLUSTER}, created by this notebook on {KIND_MARKER.read_text()}"
    )
else:
    kind_config = {
        "kind": "Cluster",
        "apiVersion": "kind.x-k8s.io/v1alpha4",
        "nodes": [{"role": "control-plane"}] + [{"role": "worker"}] * WORKERS,
    }
    sh(
        [
            "kind",
            "create",
            "cluster",
            "--name",
            KIND_CLUSTER,
            "--config",
            "-",
            "--image",
            NODE_IMAGE,
            "--wait",
            "10m",
        ],
        input=json.dumps(kind_config),
        timeout=1800,
    )
    KIND_MARKER.write_text(now_iso())
kubectl("wait", "--for=condition=Ready", "node", "--all", "--timeout=600s", timeout=700)
# 10 Deployments of pause pods, one replica per worker, with requests and limits so kubeletstats
# emits the utilization metrics
kubectl("create", "namespace", "shop", check=False)
for app in [
    "cart",
    "catalog",
    "checkout",
    "currency",
    "email",
    "frontend",
    "payment",
    "recommendation",
    "shipping",
    "ads",
]:
    labels = {"app": app}
    kubectl(
        "apply",
        "-f",
        "-",
        input=json.dumps(
            {
                "apiVersion": "apps/v1",
                "kind": "Deployment",
                "metadata": {"name": app, "namespace": "shop"},
                "spec": {
                    "replicas": WORKERS,
                    "selector": {"matchLabels": labels},
                    "template": {
                        "metadata": {"labels": labels},
                        "spec": {
                            "topologySpreadConstraints": [
                                {
                                    "maxSkew": 1,
                                    "topologyKey": "kubernetes.io/hostname",
                                    "whenUnsatisfiable": "ScheduleAnyway",
                                    "labelSelector": {"matchLabels": labels},
                                }
                            ],
                            "containers": [
                                {
                                    "name": app,
                                    "image": "registry.k8s.io/pause:3.10",
                                    "resources": {
                                        "requests": {"cpu": "5m", "memory": "8Mi"},
                                        "limits": {"cpu": "50m", "memory": "32Mi"},
                                    },
                                }
                            ],
                        },
                    },
                },
            }
        ),
        quiet=True,
    )

expected = 10 * WORKERS
for _ in range(120):
    running = kubectl(
        "get",
        "pods",
        "-n",
        "shop",
        "--no-headers",
        "--field-selector=status.phase=Running",
        quiet=True,
    ).splitlines()
    if len(running) >= expected:
        break
    time.sleep(10)

nodes = kubectl("get", "nodes", "--no-headers", quiet=True).splitlines()
workers = [n for n in nodes if "control-plane" not in n]
print(
    f"nodes: {len(nodes)} ({len(workers)} workers) | shop pods running: {len(running)}"
)
per_node = {}
for line in kubectl(
    "get",
    "pods",
    "-n",
    "shop",
    "-o",
    "jsonpath={range .items[*]}{.spec.nodeName}{'\\n'}{end}",
    quiet=True,
).split():
    per_node[line] = per_node.get(line, 0) + 1
print(
    f"shop pods per worker: min {min(per_node.values())}, max {max(per_node.values())}"
)

## 3. Install the collectors with the kube-stack Helm chart

Creates the secret with the OTLP endpoint and ingest key, and defines `helm_install()`: the EDOT
values, `values/lab.yaml` (kind and measurement settings only), the layout file and the run's values.

In [ ]:
kubectl("create", "namespace", NS, check=False)
secret = {
    "apiVersion": "v1",
    "kind": "Secret",
    "metadata": {"name": "elastic-secret-otel", "namespace": NS},
    "type": "Opaque",
    "stringData": {
        "elastic_otlp_endpoint": OTLP_ENDPOINT,
        "elastic_api_key": INGEST_API_KEY,
    },
}
kubectl("apply", "-f", "-", input=json.dumps(secret))
sh(
    [
        "helm",
        "repo",
        "add",
        "open-telemetry",
        "https://open-telemetry.github.io/opentelemetry-helm-charts",
        "--force-update",
    ]
)
sh(["helm", "repo", "update", "open-telemetry"])


def helm_install(layout, run_values_file):
    helm(
        "upgrade",
        "--install",
        "opentelemetry-kube-stack",
        "open-telemetry/opentelemetry-kube-stack",
        "--namespace",
        NS,
        "--version",
        CHART_VERSION,
        "--values",
        EDOT_VALUES,
        "--values",
        "values/lab.yaml",
        *[arg for f in LAYOUT_FILES[layout] for arg in ("--values", f)],
        "--values",
        str(run_values_file),
        quiet=True,
        timeout=900,
    )

## 4. Send each run to its own data stream

Sets the collection interval and writes the run id (`l3s10` = layout 3, 10 seconds) into
`data_stream.namespace`.

In [ ]:
def run_values(run, interval):
    return f"""clusterName: {KIND_CLUSTER}
collectors:
  daemon:
    config:
      receivers:
        hostmetrics: {{collection_interval: {interval}s}}
        kubeletstats: {{collection_interval: {interval}s}}
      processors:
        resource/run:
          attributes:
            - {{key: data_stream.namespace, value: {run}, action: upsert}}
      service:
        pipelines:
          metrics/node/otel:
            processors: [batch/metrics, k8s_attributes, resourcedetection/env, resourcedetection/system,
                         resource/hostname, resource/cloud, resource/run]
"""


print(run_values(f"l3s10{SUFFIX}", 10))

## 5. Remove the gateway for the DaemonSet-only layout

In [ ]:
print(Path("values/layout1-daemonset-only.yaml").read_text())

## 6. Filter and aggregate metrics in the gateway

In [ ]:
print(Path("values/layout3-gateway-filter.yaml").read_text())

## 7. Run each layout at each interval

Installs each run in `LAB_RUNS`, warms up for `WARMUP_SECONDS` and measures for `MEASURE_SECONDS`.
About 25 minutes per run. Results go to `results/<run>/`.

In [ ]:
PROBES = json.dumps(
    {
        "spec": {
            "startupProbe": {
                "failureThreshold": 24,
                "periodSeconds": 5,
                "timeoutSeconds": 5,
            },
            "readinessProbe": {
                "periodSeconds": 10,
                "timeoutSeconds": 5,
                "failureThreshold": 6,
            },
        }
    }
)


def collector_pods():
    out = kubectl(
        "-n",
        NS,
        "get",
        "pods",
        "-l",
        "app.kubernetes.io/component=opentelemetry-collector",
        "-o",
        "name",
        quiet=True,
    )
    return [p.split("/", 1)[1] for p in out.split()]


def scrape(out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    failed = []
    for pod in collector_pods():
        p = subprocess.run(
            [
                "kubectl",
                "--context",
                KCTX,
                "get",
                "--raw",
                f"/api/v1/namespaces/{NS}/pods/{pod}:8888/proxy/metrics",
            ],
            capture_output=True,
            text=True,
            timeout=60,
        )
        if p.returncode == 0:
            (out_dir / f"{pod}.prom").write_text(p.stdout)
        else:
            failed.append(pod)
    return failed


def run_layout(run, layout, interval):
    d = RESULTS / run
    d.mkdir(parents=True, exist_ok=True)
    (d / "run-values.yaml").write_text(run_values(run, interval))
    print(
        f"{now_iso()} {run}: helm upgrade, layout {layout} ({LAYOUT_NAMES[layout]}), {interval}s"
    )
    helm_install(layout, d / "run-values.yaml")
    for c in ("daemon", "gateway"):
        kubectl(
            "-n",
            NS,
            "patch",
            "opentelemetrycollector",
            f"opentelemetry-kube-stack-{c}",
            "--type",
            "merge",
            "-p",
            PROBES,
            check=False,
            quiet=True,
        )
    for _ in range(60):
        if kubectl(
            "-n",
            NS,
            "get",
            "ds/opentelemetry-kube-stack-daemon-collector",
            check=False,
            quiet=True,
        ):
            break
        time.sleep(10)
    time.sleep(20)
    kubectl(
        "-n",
        NS,
        "rollout",
        "status",
        "ds/opentelemetry-kube-stack-daemon-collector",
        "--timeout=1800s",
        quiet=True,
        timeout=1900,
    )
    if layout != 1:
        kubectl(
            "-n",
            NS,
            "rollout",
            "status",
            "deploy/opentelemetry-kube-stack-gateway-collector",
            "--timeout=1800s",
            quiet=True,
            timeout=1900,
        )

    pods = collector_pods()
    inventory = {
        "nodes": len(kubectl("get", "nodes", "--no-headers", quiet=True).splitlines()),
        "pods_running_total": len(
            kubectl(
                "get",
                "pods",
                "-A",
                "--no-headers",
                "--field-selector=status.phase=Running",
                quiet=True,
            ).splitlines()
        ),
        "pods_running_shop": len(
            kubectl(
                "get",
                "pods",
                "-n",
                "shop",
                "--no-headers",
                "--field-selector=status.phase=Running",
                quiet=True,
            ).splitlines()
        ),
        "daemonset_desired": int(
            kubectl(
                "-n",
                NS,
                "get",
                "ds/opentelemetry-kube-stack-daemon-collector",
                "-o",
                "jsonpath={.status.desiredNumberScheduled}",
                quiet=True,
            )
        ),
        "collector_pods": {
            role: sum(role in p for p in pods)
            for role in ("daemon", "cluster-stats", "gateway")
        },
    }
    (d / "inventory.json").write_text(json.dumps(inventory, indent=1))
    print(f"{now_iso()} {run}: rolled out {inventory}; warm-up {WARMUP}s")
    time.sleep(WARMUP)

    t0 = now_iso()
    failed0 = scrape(d / "selfmetrics-t0")
    time.sleep(MEASURE)
    t1 = now_iso()
    failed1 = scrape(d / "selfmetrics-t1")
    (d / "t0").write_text(t0)
    (d / "t1").write_text(t1)
    print(
        f"{t1} {run}: done, window {t0} -> {t1}, scrape failures t0={len(failed0)} t1={len(failed1)}"
    )


for run, layout, interval in RUNS:
    if (RESULTS / run / "t1").exists():
        print(
            f"{run}: already measured, skipping (delete results/{run} to run it again)"
        )
        continue
    if (RESULTS / run / "run-values.yaml").exists():
        print(
            f"WARNING: {run} was interrupted earlier; its data stream already has documents from that attempt. "
            f"For clean numbers, delete the {run} data streams and results/{run}, then rerun."
        )
    run_layout(run, layout, interval)

## 8. Stop the collectors

Uninstalls the chart and deletes the kind cluster (`KEEP_KIND_CLUSTER="true"` keeps it), so the
last run's data streams stop receiving data before `_forcemerge`.

In [ ]:
def stop_collectors():
    """Delete the collectors while the operator still runs (it removes their finalizers), then
    uninstall the chart. With Helm 4, `helm uninstall --wait` can hang on the collector resources.
    """
    if "opentelemetry-kube-stack" not in helm(
        "list", "--namespace", NS, "--short", check=False, quiet=True
    ):
        print("no collectors installed")
        return
    kubectl(
        "-n",
        NS,
        "delete",
        "opentelemetrycollectors",
        "--all",
        "--timeout=300s",
        check=False,
        timeout=360,
    )
    helm(
        "uninstall",
        "opentelemetry-kube-stack",
        "--namespace",
        NS,
        "--timeout",
        "120s",
        check=False,
        timeout=300,
    )
    for _ in range(30):
        if not collector_pods():
            break
        time.sleep(10)
    print("collector pods left:", len(collector_pods()))


stop_collectors()
time.sleep(60)  # let the export queues drain
if KEEP_KIND_CLUSTER:
    print(f"kind cluster {KIND_CLUSTER} kept")
else:
    sh(["kind", "delete", "cluster", "--name", KIND_CLUSTER], timeout=600)
    KIND_MARKER.unlink(missing_ok=True)

## 9. Count the pod network interfaces per node

In [ ]:
unfiltered = [r for r, layout, _ in RUNS if layout != 3]
NET_RUN = (
    f"l2s10{SUFFIX}"
    if f"l2s10{SUFFIX}" in RUN_IDS
    else (unfiltered[0] if unfiltered else None)
)
if NET_RUN is None:
    print(
        "no layout 1 or 2 run in LAB_RUNS: layout 3 drops the veth data points, nothing to count"
    )
else:
    base = f"""FROM metrics-hostmetricsreceiver.otel-{NET_RUN}
| WHERE metrics.system.network.io IS NOT NULL
| EVAL veth = CASE(attributes.device LIKE "veth*", attributes.device, null)
| STATS network_devices = COUNT_DISTINCT(attributes.device), veth_devices = COUNT_DISTINCT(veth) BY host.name"""
    esql(
        base + "\n| SORT host.name\n| LIMIT 5",
        f"Network devices per node ({NET_RUN}), first 5 hosts",
    )
    rows = esql_rows(base + "\n| LIMIT 100")
    devs = [r["network_devices"] for r in rows]
    veth = [r["veth_devices"] for r in rows]
    shares = [v / d for v, d in zip(veth, devs)]
    print(
        f"{len(rows)} hosts: {min(devs)} to {max(devs)} network devices, {min(veth)} to {max(veth)} veth, "
        f"veth share {min(shares):.0%} to {max(shares):.0%}"
    )

## 10. Measure data points and storage per day

In [ ]:
PROM_LINE = re.compile(r"^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{[^}]*\})?\s+([-+0-9.eE]+|NaN)$")


def load_prom(path):
    out = {}
    for line in path.read_text().splitlines():
        m = PROM_LINE.match(line)
        if m:
            labels = dict(re.findall(r'(\w+)="([^"]*)"', m.group(2) or ""))
            out.setdefault(m.group(1), []).append((labels, float(m.group(3))))
    return out


def prom_total(metrics, name, **match):
    return sum(
        v
        for lab, v in metrics.get(name, [])
        if all(re.fullmatch(p, lab.get(k, "")) for k, p in match.items())
    )


def role_of(pod):
    return next(
        (r for r in ("daemon", "gateway", "cluster-stats") if r in pod), "other"
    )


SELF_COUNTERS = [
    (
        "hostmetrics_points",
        "otelcol_receiver_accepted_metric_points",
        {"receiver": "hostmetrics"},
    ),
    (
        "kubeletstats_points",
        "otelcol_receiver_accepted_metric_points",
        {"receiver": "kubeletstats"},
    ),
    (
        "sent_to_elastic_points",
        "otelcol_exporter_sent_metric_points",
        {"exporter": r"otlp_grpc/ingest.*"},
    ),
    ("failed_points", "otelcol_exporter_send_failed_metric_points", {}),
    ("cpu_seconds", "otelcol_process_cpu_seconds", {}),
    (
        "metric_export_calls_to_elastic",
        "rpc_client_call_duration_count",
        {"server_port": "443", "rpc_method": r".*MetricsService/Export"},
    ),
    (
        "all_export_calls_to_elastic",
        "rpc_client_call_duration_count",
        {"server_port": "443"},
    ),
]


def selfmetrics(run):
    d = RESULTS / run
    t0, t1 = window(run)
    secs = (parse_iso(t1) - parse_iso(t0)).total_seconds()
    roles, senders, restarted = {}, 0, 0
    for f1 in sorted((d / "selfmetrics-t1").glob("*.prom")):
        f0 = d / "selfmetrics-t0" / f1.name
        if not f0.exists():
            continue
        m0, m1 = load_prom(f0), load_prom(f1)
        a = roles.setdefault(role_of(f1.stem), {"pods": 0, "rss_mib": []})
        a["pods"] += 1
        if prom_total(m1, "otelcol_process_uptime") < prom_total(
            m0, "otelcol_process_uptime"
        ):
            restarted += 1
        for key, name, match in SELF_COUNTERS:
            a[key] = (
                a.get(key, 0.0)
                + prom_total(m1, name, **match)
                - prom_total(m0, name, **match)
            )
        a["rss_mib"].append(prom_total(m1, "otelcol_process_memory_rss") / 2**20)
        if (
            prom_total(m1, "rpc_client_call_duration_count", server_port="443")
            - prom_total(m0, "rpc_client_call_duration_count", server_port="443")
            > 0
        ):
            senders += 1
    out = {
        "seconds": secs,
        "restarted_pods": restarted,
        "pods_sending_to_elastic": senders,
        "roles": {},
    }
    for r, a in roles.items():
        rss = a.pop("rss_mib")
        o = {"pods": a.pop("pods"), "rss_mib_per_pod": round(sum(rss) / len(rss), 1)}
        o.update({f"{k}_per_s": round(v / secs, 2) for k, v in a.items()})
        o["cpu_millicores_total"] = round(a["cpu_seconds"] / secs * 1000, 1)
        out["roles"][r] = o
    return out


def storage(run, ds, client=es, stream=None, t=None):
    stream = stream or f"metrics-{ds}-{run}"
    cache = RESULTS / run / f"storage-{stream}.json"
    if cache.exists():
        return json.loads(cache.read_text())
    t0, t1 = t or window(run)
    secs = (parse_iso(t1) - parse_iso(t0)).total_seconds()
    c = client.options(request_timeout=1800)
    idx = [
        i["index_name"]
        for i in c.indices.get_data_stream(name=stream)["data_streams"][0]["indices"]
    ]
    rng = {"range": {"@timestamp": {"gte": t0, "lt": t1}}}
    docs_window = c.count(index=stream, query=rng)["count"]
    caps = c.field_caps(index=stream, fields="metrics.*")["fields"]
    metric_fields = sorted(
        f
        for f, types in caps.items()
        if any(v.get("time_series_metric") for v in types.values())
    )
    aggs = {f"m{i}": {"value_count": {"field": f}} for i, f in enumerate(metric_fields)}
    points_window = int(
        sum(
            v["value"]
            for v in c.search(index=stream, size=0, query=rng, aggs=aggs)[
                "aggregations"
            ].values()
        )
    )
    series = c.search(
        index=stream,
        size=0,
        query=rng,
        aggs={"s": {"cardinality": {"field": "_tsid", "precision_threshold": 40000}}},
    )["aggregations"]["s"]["value"]
    total_docs = sum(c.count(index=i)["count"] for i in idx)
    c.indices.forcemerge(index=",".join(idx), max_num_segments=1)
    du = c.indices.disk_usage(index=",".join(idx), run_expensive_tasks=True).body
    time.sleep(45)  # let the pre-merge segment files be deleted
    disk = sum(v["store_size_in_bytes"] for k, v in du.items() if k != "_shards")
    fields = {}
    for k, v in du.items():
        if k != "_shards":
            for f, fv in v["fields"].items():
                fields[f] = fields.get(f, 0) + fv["total_in_bytes"]
    bpd = disk / total_docs
    res = {
        "stream": stream,
        "indices": idx,
        "window": [t0, t1],
        "window_seconds": secs,
        "docs_window": docs_window,
        "docs_per_s": round(docs_window / secs, 2),
        "points_window": points_window,
        "points_per_s": round(points_window / secs, 2),
        "series_window": series,
        "total_docs": total_docs,
        "disk_usage_bytes": disk,
        "source_bytes": fields.get("_source", 0),
        "id_bytes": fields.get("_id", 0),
        "bytes_per_doc": round(bpd, 1),
        "gib_per_day": round(bpd * docs_window / secs * 86400 / 2**30, 3),
    }
    cache.write_text(json.dumps(res, indent=1))
    return res


SELF, STORAGE = {}, {}
for run, layout, interval in RUNS:
    SELF[run] = selfmetrics(run)
    STORAGE[run] = {ds: storage(run, ds) for ds in DATASETS}
    (RESULTS / run / "selfmetrics.json").write_text(json.dumps(SELF[run], indent=1))

print(
    "| Run | Layout | Interval | Collected points/s | Stored points/s (hm + kb) | Stored vs collected | hostmetrics GiB/day | kubeletstats GiB/day | Total GiB/day | _source share | Failed points/s | Restarts |"
)
print("|---|---|---|---|---|---|---|---|---|---|---|---|")
GIB, POINTS, COLLECTED = {}, {}, {}
for run, layout, interval in RUNS:
    d = SELF[run]["roles"].get("daemon", {})
    collected = d.get("hostmetrics_points_per_s", 0) + d.get(
        "kubeletstats_points_per_s", 0
    )
    hm, kb = (
        STORAGE[run]["hostmetricsreceiver.otel"],
        STORAGE[run]["kubeletstatsreceiver.otel"],
    )
    stored = hm["points_per_s"] + kb["points_per_s"]
    GIB[run], POINTS[run], COLLECTED[run] = (
        hm["gib_per_day"] + kb["gib_per_day"],
        stored,
        collected,
    )
    share = [s["source_bytes"] / s["disk_usage_bytes"] for s in (hm, kb)]
    failed = sum(r.get("failed_points_per_s", 0) for r in SELF[run]["roles"].values())
    vs = f"{pct(stored, collected):+.1f}%" if layout != 3 else "filtered"
    print(
        f"| {run} | {LAYOUT_NAMES[layout]} | {interval}s | {collected:,.0f} | {stored:,.0f} | {vs} | {hm['gib_per_day']:.2f} | {kb['gib_per_day']:.2f} | "
        f"{GIB[run]:.2f} | {min(share):.0%} to {max(share):.0%} | {failed} | {SELF[run]['restarted_pods']} |"
    )

by = {(layout, interval): run for run, layout, interval in RUNS}
print("\nCollected data points per second by interval (average over layouts):")
for interval in sorted({i for _, _, i in RUNS}, reverse=True):
    vals = [COLLECTED[r] for (l, i), r in by.items() if i == interval]
    print(f"  {interval}s: {statistics.mean(vals):,.0f}")
print("\nInterval effect:")
for layout in (1, 2, 3):
    if (layout, 10) in by and (layout, 30) in by:
        g10, g30 = GIB[by[layout, 10]], GIB[by[layout, 30]]
        print(
            f"  layout {layout}: 30s -> 10s x{g10 / g30:.2f}, 10s -> 30s {pct(g30, g10):+.1f}%"
        )
print("\nLayout 1 vs layout 2 (storage):")
for interval in (60, 30, 10):
    if (1, interval) in by and (2, interval) in by:
        print(f"  {interval}s: {pct(GIB[by[1, interval]], GIB[by[2, interval]]):+.1f}%")
print("\nFilter (layout 3) vs layout 2 and layout 1:")
for interval in (60, 30, 10):
    if (3, interval) in by:
        l3 = by[3, interval]
        for ref in (2, 1):
            if (ref, interval) in by:
                r = by[ref, interval]
                hm3, hmr = (
                    STORAGE[l3]["hostmetricsreceiver.otel"]["points_per_s"],
                    STORAGE[r]["hostmetricsreceiver.otel"]["points_per_s"],
                )
                print(
                    f"  {interval}s vs layout {ref}: hostmetrics data points {pct(hm3, hmr):+.1f}%, storage {pct(GIB[l3], GIB[r]):+.1f}%"
                )

## 11. Check which `_source` mode the run indices use

In [ ]:
modes = {}
for run in RUN_IDS:
    for ds in DATASETS:
        for idx in STORAGE[run][ds]["indices"]:
            s = es.indices.get_settings(
                index=idx,
                name="index.mode,index.mapping.source.mode",
                include_defaults=True,
                flat_settings=True,
            )[idx]
            flat = {**s.get("defaults", {}), **s.get("settings", {})}
            key = (
                flat.get("index.mode"),
                flat.get("index.mapping.source.mode"),
                (
                    "explicit"
                    if "index.mapping.source.mode" in s.get("settings", {})
                    else "default"
                ),
            )
            modes[key] = modes.get(key, 0) + 1
for (mode, source_mode, origin), n in modes.items():
    print(
        f"{n} backing indices: index.mode={mode}, index.mapping.source.mode={source_mode!r} ({origin})"
    )

## 12. Optional: measure the same data with synthetic `_source`

Runs only with `RUN_SYNTHETIC="true"`, after section 8: copies the runs into a local Elasticsearch
9.5.5 trial in Docker (6 GiB) and measures them again.

In [ ]:
SYN = {}
if not RUN_SYNTHETIC:
    print("RUN_SYNTHETIC is not true: skipped")
else:
    src = urlparse(ES_URL)
    src_host = f"{src.hostname}:{src.port or 443}"
    password = secrets.token_urlsafe(16)
    sh(
        ["docker", "rm", "-f", SYN_CONTAINER], check=False, quiet=True
    )  # a copy left by an earlier run
    sh(
        [
            "docker",
            "run",
            "-d",
            "--name",
            SYN_CONTAINER,
            "-p",
            f"127.0.0.1:{SYN_PORT}:9200",
            "-m",
            "6g",
            "-e",
            "discovery.type=single-node",
            "-e",
            "xpack.security.enabled=true",
            "-e",
            "xpack.security.http.ssl.enabled=false",
            "-e",
            "ES_JAVA_OPTS=-Xms3g -Xmx3g",
            "-e",
            f"ELASTIC_PASSWORD={password}",
            "-e",
            f"reindex.remote.whitelist={src_host}",
            "docker.elastic.co/elasticsearch/elasticsearch:9.5.5",
        ],
        quiet=True,
    )
    dest = Elasticsearch(
        f"http://localhost:{SYN_PORT}",
        basic_auth=("elastic", password),
        request_timeout=120,
    )
    for _ in range(60):
        try:
            if dest.cluster.health(wait_for_status="yellow", timeout="5s")[
                "status"
            ] in ("yellow", "green"):
                break
        except Exception:
            time.sleep(5)
    dest.license.post_start_trial(acknowledge=True)
    print(
        "destination:",
        dest.info()["version"]["number"],
        dest.license.get()["license"]["type"],
    )

    # Pin the metric field mappings of the source runs, accept documents as old as the first run
    props, metrics_obj = {}, None
    for run in RUN_IDS:
        for ds in DATASETS:
            for m in es.indices.get_mapping(index=f"metrics-{ds}-{run}").body.values():
                metrics = m["mappings"]["properties"]["metrics"]
                metrics_obj = metrics_obj or {
                    k: v for k, v in metrics.items() if k != "properties"
                }
                for field, spec in metrics.get("properties", {}).items():
                    if field in props and props[field] != spec:
                        raise RuntimeError(f"conflicting mapping for {field}")
                    props[field] = spec
    metrics_obj["properties"] = props
    oldest = min(parse_iso(window(r)[0]) for r in RUN_IDS)
    look_back_h = int((datetime.now(timezone.utc) - oldest).total_seconds() // 3600) + 6
    dest.cluster.put_component_template(
        name="metrics-otel@custom",
        template={
            "settings": {"index.look_back_time": f"{look_back_h}h"},
            "mappings": {"properties": {"metrics": metrics_obj}},
        },
    )
    print(f"pinned {len(props)} metrics.* fields, look_back_time {look_back_h}h")

    tasks = {}
    for run in RUN_IDS:
        for ds in DATASETS:
            t = dest.reindex(
                source={
                    "remote": {
                        "host": f"https://{src_host}",
                        "headers": {"Authorization": f"ApiKey {API_KEY}"},
                    },
                    "index": f"metrics-{ds}-{run}",
                    "size": 2000,
                },
                dest={"index": f"metrics-{ds}-s{run}", "op_type": "create"},
                wait_for_completion=False,
            )
            tasks[(run, ds)] = t["task"]
    while tasks:
        time.sleep(15)
        for k, t in list(tasks.items()):
            st = dest.tasks.get(task_id=t)
            if st["completed"]:
                r = st.get("response", {})
                if r.get("failures"):
                    raise RuntimeError(f"reindex {k}: {r['failures'][0]}")
                print(f"copied {k[0]} {k[1]}: {r.get('created')} documents")
                del tasks[k]

    print("\n| Run | Stored GiB/day | Synthetic GiB/day | Synthetic / stored |")
    print("|---|---|---|---|")
    for run, layout, interval in RUNS:
        syn_gib = 0.0
        for ds in DATASETS:
            stored = STORAGE[run][ds]
            copy = storage(
                run, ds, client=dest, stream=f"metrics-{ds}-s{run}", t=window(run)
            )
            # compare without _id, see above
            ratio = (copy["disk_usage_bytes"] - copy["id_bytes"]) / (
                stored["disk_usage_bytes"] - stored["id_bytes"]
            )
            syn_gib += stored["gib_per_day"] * ratio
        SYN[run] = round(syn_gib, 3)
        print(
            f"| {run} | {GIB[run]:.2f} | {SYN[run]:.2f} | {SYN[run] / GIB[run]:.0%} |"
        )

## 13. Measure what the collectors themselves cost

In [ ]:
COST = {}
for run, layout, interval in RUNS:
    t0, t1 = window(run)
    COST[run] = {
        r["role"]: r
        for r in esql_rows(
            f"""FROM metrics-kubeletstatsreceiver.otel-{run}
| WHERE @timestamp >= "{t0}" AND @timestamp < "{t1}" AND k8s.namespace.name == "{NS}"
| EVAL role = CASE(k8s.pod.name LIKE "*daemon*", "node collector", k8s.pod.name LIKE "*gateway*", "gateway", k8s.pod.name LIKE "*cluster-stats*", "cluster collector", "operator")
| STATS working_set_mib = AVG(metrics.k8s.pod.memory.working_set) / 1048576, cpu_millicores = AVG(metrics.k8s.pod.cpu.usage) * 1000, pods = COUNT_DISTINCT(k8s.pod.name) BY role
| LIMIT 10"""
        )
    }
    (RESULTS / run / "collector-cost.json").write_text(json.dumps(COST[run], indent=1))

print(
    "| Run | Collector pods (node, cluster, gateway) | Pods sending to Elastic | Memory per node collector | Memory of all gateway pods | Metric export calls to Elastic/s |"
)
print("|---|---|---|---|---|---|")
for run, layout, interval in RUNS:
    inv = json.loads((RESULTS / run / "inventory.json").read_text())["collector_pods"]
    node = COST[run].get("node collector", {})
    gw = COST[run].get("gateway", {})
    gw_total = (gw.get("working_set_mib") or 0) * (gw.get("pods") or 0)
    calls = sum(
        r.get("metric_export_calls_to_elastic_per_s", 0)
        for r in SELF[run]["roles"].values()
    )
    print(
        f"| {run} | {sum(inv.values())} ({inv['daemon']}, {inv['cluster-stats']}, {inv['gateway']}) | {SELF[run]['pods_sending_to_elastic']} | "
        f"{node.get('working_set_mib', 0):.0f} MiB | {gw_total:.0f} MiB | {calls:.1f} |"
    )

for layout in (1, 2, 3):
    runs = [r for r, l, _ in RUNS if l == layout]
    if runs:
        node = [
            COST[r]["node collector"]["working_set_mib"]
            for r in runs
            if "node collector" in COST[r]
        ]
        line = f"layout {layout}: node collector {min(node):.0f} to {max(node):.0f} MiB, all {WORKERS} together {min(node) * WORKERS / 1024:.1f} to {max(node) * WORKERS / 1024:.1f} GiB"
        gws = [
            COST[r]["gateway"]["working_set_mib"] * COST[r]["gateway"]["pods"]
            for r in runs
            if "gateway" in COST[r]
        ]
        if gws:
            line += f"; gateway pods together {min(gws):.0f} to {max(gws):.0f} MiB"
        print(line)

run = f"l2s10{SUFFIX}" if f"l2s10{SUFFIX}" in RUN_IDS else RUN_IDS[0]
t0, t1 = window(run)
_ = esql(
    f"""FROM metrics-kubeletstatsreceiver.otel-{run}
| WHERE @timestamp >= "{t0}" AND @timestamp < "{t1}" AND k8s.namespace.name == "{NS}"
| EVAL role = CASE(k8s.pod.name LIKE "*daemon*", "node collector", k8s.pod.name LIKE "*gateway*", "gateway", k8s.pod.name LIKE "*cluster-stats*", "cluster collector", "operator")
| STATS pods = COUNT_DISTINCT(k8s.pod.name), avg_working_set_mib = ROUND(AVG(metrics.k8s.pod.memory.working_set) / 1048576), avg_cpu_millicores = ROUND(AVG(metrics.k8s.pod.cpu.usage) * 1000) BY role
| SORT pods DESC
| LIMIT 10""",
    f"Collector cost by role ({run}, the article's Discover query)",
)

## 14. Time the Hosts view and Inventory requests

Replays Kibana's internal requests for both views; their bodies can change between versions.

In [ ]:
def kbn_post(path, body):
    for attempt in range(5):
        t = time.perf_counter()
        r = requests.post(
            f"{KBN_BASE}{path}", headers=KBN_HEADERS, json=body, timeout=120
        )
        ms = (time.perf_counter() - t) * 1000
        if r.status_code < 500 or attempt == 4:
            r.raise_for_status()
            return ms, r.json()
        time.sleep(3)


def timed(fn):
    for _ in range(2):
        fn()
    times, last = [], None
    for _ in range(LATENCY_REPS):
        ms, last = fn()
        times.append(ms)
    return statistics.median(times), last


def status():
    t = time.perf_counter()
    r = requests.get(f"{KIBANA_URL}/api/status", headers=KBN_HEADERS, timeout=60)
    r.raise_for_status()
    return (time.perf_counter() - t) * 1000, None


baseline, _ = timed(status)
print(f"Kibana status call: median {baseline:.0f} ms\n")
print(
    "| Run | Hosts view median | Inventory median | Query time (median - status) | Hosts returned (Hosts view / Inventory) | Hosts with empty or 0 Disk Usage | Normalized load (median) |"
)
print("|---|---|---|---|---|---|---|")
LATENCY = {}
for run, layout, interval in RUNS:
    t0, t1 = window(run)
    to_ms = lambda s: int(parse_iso(s).timestamp() * 1000)
    hosts_body = {
        "query": {
            "bool": {
                "must": [],
                "filter": [{"term": {"data_stream.namespace": run}}],
                "should": [],
                "must_not": [],
            }
        },
        "from": t0,
        "to": t1,
        "limit": 100,
        "schema": "semconv",
        "metrics": [
            "cpuV2",
            "diskSpaceUsage",
            "memory",
            "memoryFree",
            "normalizedLoad1m",
        ],
    }
    inv_body = {
        "accountId": "",
        "dropPartialBuckets": True,
        "kuery": f'data_stream.namespace : "{run}"',
        "groupBy": [],
        "includeTimeseries": True,
        "metrics": [{"type": "cpuV2"}],
        "nodeType": "host",
        "sourceId": "default",
        "region": "",
        "schema": "semconv",
        "timerange": {
            "interval": "1m",
            "to": to_ms(t1),
            "from": to_ms(t0),
            "lookbackSize": 5,
        },
    }
    hv_ms, hv = timed(lambda: kbn_post("/api/metrics/infra/host", hosts_body))
    inv_ms, inv = timed(lambda: kbn_post("/api/metrics/snapshot", inv_body))
    hosts = hv.get("nodes", hv.get("hosts", []))  # Kibana 9.5 returns "nodes"
    metric = lambda h, name: next(
        (m.get("value") for m in h.get("metrics", []) if m.get("name") == name), None
    )
    no_disk = sum(
        not metric(h, "diskSpaceUsage") for h in hosts
    )  # null or 0: shown as N/A / 0% in the Hosts view
    loads = [
        metric(h, "normalizedLoad1m")
        for h in hosts
        if metric(h, "normalizedLoad1m") is not None
    ]
    LATENCY[run] = {
        "hosts_view_ms": round(hv_ms),
        "inventory_ms": round(inv_ms),
        "status_ms": round(baseline),
        "hosts": len(hosts),
        "inventory_nodes": len(inv.get("nodes", [])),
        "hosts_without_disk_usage": no_disk,
    }
    (RESULTS / run / "latency.json").write_text(json.dumps(LATENCY[run], indent=1))
    load = f"{statistics.median(loads):.0%}" if loads else "n/a"
    print(
        f"| {run} | {hv_ms:.0f} ms | {inv_ms:.0f} ms | {hv_ms - baseline:.0f} / {inv_ms - baseline:.0f} ms | "
        f"{len(hosts)} / {len(inv.get('nodes', []))} | {no_disk} | {load} |"
    )

## 15. Find where cardinality comes from

In [ ]:
print(
    "| Run | hostmetrics series | kubeletstats series | kubeletstats share of storage | Data streams in the namespace |"
)
print("|---|---|---|---|---|")
for run, layout, interval in RUNS:
    hm, kb = (
        STORAGE[run]["hostmetricsreceiver.otel"],
        STORAGE[run]["kubeletstatsreceiver.otel"],
    )
    streams = es.indices.get_data_stream(name=f"*-{run}")["data_streams"]
    print(
        f"| {run} | {hm['series_window']:,} | {kb['series_window']:,} | {kb['gib_per_day'] / GIB[run]:.0%} | {len(streams)} ({', '.join(s['name'] for s in streams)}) |"
    )

for interval in (60, 30, 10):
    if (3, interval) in by and (2, interval) in by:
        s3 = STORAGE[by[3, interval]]["hostmetricsreceiver.otel"]["series_window"]
        s2 = STORAGE[by[2, interval]]["hostmetricsreceiver.otel"]["series_window"]
        print(
            f"{interval}s: filter changed hostmetrics series by {pct(s3, s2):+.1f}% against layout 2"
        )

## 16. Summarize the runs

Also written to `results/results.json`.

In [ ]:
rows = []
for run, layout, interval in RUNS:
    rows.append(
        {
            "run": run,
            "layout": LAYOUT_NAMES[layout],
            "interval_s": interval,
            "collected_points_per_s": round(COLLECTED[run], 1),
            "stored_points_per_s": round(POINTS[run], 1),
            "gib_per_day": round(GIB[run], 3),
            "gib_per_day_synthetic": SYN.get(run),
            "node_collector_mib": round(
                COST[run].get("node collector", {}).get("working_set_mib") or 0
            ),
            "hosts_view_ms": LATENCY[run]["hosts_view_ms"],
            "inventory_ms": LATENCY[run]["inventory_ms"],
            "restarted_pods": SELF[run]["restarted_pods"],
        }
    )
(RESULTS / "results.json").write_text(json.dumps(rows, indent=1))
cols = list(rows[0])
print("| " + " | ".join(cols) + " |")
print("|" + "---|" * len(cols))
for r in rows:
    print("| " + " | ".join(str(r[c]) for c in cols) + " |")

## 17. Clean up

Deletes the run data streams, the Helm release, the kind cluster and the synthetic `_source`
container. The two API keys stay: invalidate them in Kibana > Stack Management > API keys.

In [ ]:
for run in RUN_IDS:
    for s in (
        es.options(ignore_status=404)
        .indices.get_data_stream(name=f"*-{run}")
        .body.get("data_streams", [])
    ):
        es.indices.delete_data_stream(name=s["name"])
        print("deleted data stream", s["name"])
if KIND_CLUSTER in sh(["kind", "get", "clusters"], quiet=True).split():
    sh(
        ["kind", "delete", "cluster", "--name", KIND_CLUSTER], timeout=600
    )  # removes the collectors too
KIND_MARKER.unlink(missing_ok=True)
sh(["docker", "rm", "-f", SYN_CONTAINER], check=False, quiet=True)
print("done")